In [1]:
print("Shubham Singh")

Shubham Singh


In [2]:
## Load the data


import json

with open("../data/filings.json") as f:
    docs = json.load(f)

In [3]:
print(docs[0])

{'doc_id': 'ARS-Q2FY26-RES-01', 'company': 'Aarav Steel Ltd', 'ticker': 'AARAVSTL', 'doc_type': 'Quarterly Results', 'date': '2026-08-05', 'quarter': 'Q1FY27', 'section': 'Financial Highlights', 'text': 'Aarav Steel Ltd reported consolidated revenue of INR 4,812 crore for Q1FY27, up 11.4% year-on-year, driven by higher realizations in flat steel products. EBITDA margin expanded to 17.2% from 15.8% in the prior year quarter, aided by softer coking coal costs.'}


In [4]:
# Now we need to build a version for finding the answer 

#1. WE need the vectorizer let us use the tfIDF

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words= "english")
# now to fit it  we need text

texts = [d["text"] for d in docs]

In [5]:
# fir the vecotrizer on the text

matrix = vectorizer.fit_transform(texts)

In [6]:
matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 325 stored elements and shape (15, 232)>

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def search (query, docs, top_k = 3):
    query_vector = vectorizer.transform([query])
    sims = cosine_similarity(query_vector, matrix)[0]
    ranked = sims.argsort()[::-1][:top_k]
    return [(docs[i], sims[i]) for i in ranked]

In [8]:
print(search("What was Aarav Steel's revenue?", docs=docs))

[({'doc_id': 'ARS-Q2FY26-RES-01', 'company': 'Aarav Steel Ltd', 'ticker': 'AARAVSTL', 'doc_type': 'Quarterly Results', 'date': '2026-08-05', 'quarter': 'Q1FY27', 'section': 'Financial Highlights', 'text': 'Aarav Steel Ltd reported consolidated revenue of INR 4,812 crore for Q1FY27, up 11.4% year-on-year, driven by higher realizations in flat steel products. EBITDA margin expanded to 17.2% from 15.8% in the prior year quarter, aided by softer coking coal costs.'}, np.float64(0.4153581327167633)), ({'doc_id': 'ARS-INS-2026-09-01', 'company': 'Aarav Steel Ltd', 'ticker': 'AARAVSTL', 'doc_type': 'Insider Trading Disclosure', 'date': '2026-09-02', 'quarter': 'Q2FY27', 'section': 'SAST Disclosure', 'text': 'Promoter entity Aarav Family Trust sold 3,20,000 shares (0.42% of paid-up capital) between 25-29 August 2026 at an average price of INR 612.4, reducing promoter holding from 54.1% to 53.68%.'}, np.float64(0.10158245787053162)), ({'doc_id': 'BPH-Q2FY26-RES-01', 'company': 'Bhavani Pharma L

In [9]:
## let us make this search an answer
#THERSHOLD = 0.2
#def answer(query, docs, top_k = 3):
#    results = search(query, docs, top_k)
#    if not results or results[0][1] < THERSHOLD:
#        return "I do not have Proper Source for this"
    
#    lines = [f"{doc['text']} [{doc['doc_id']}]" for doc, score in results]
#    return "\n".join(lines)

In [10]:
# This will work but it could still show the data for wrong quarter so we need differnt function for that

In [11]:
import re

def extract_quarter(text):
    m = re.search("Q[1-4]FY\d{2}", text, re.IGNORECASE)
    
    if m:
        return m.group(0).upper()
    else:
        return None

In [12]:
print(extract_quarter("Q1FY98 earnings report"))

Q1FY98


In [13]:
## Now let us update our answer function

In [14]:
THERSHOLD = 0.2
def answer(query, docs, top_k = 3):
    results = search(query, docs, top_k)
    if not results:
        return "I don't have a reliable source for this."
    
    q_quarter = extract_quarter(query) # this get the quarter if the user id asking
    top_doc, top_score = results[0]
    
    if q_quarter and extract_quarter(top_doc.get("quarter", ""))  != q_quarter:
        top_score *= 0.4
        
    if top_score < THERSHOLD:
        return "I don't have a reliable source for this."
    
    lines = [f"{doc['text']} [{doc['doc_id']}]" for doc, score in results]
    return "\n".join(lines)

In [15]:
## Now let us evalute that are getting the right docs for the answer

with open("../data/eval_set.json", 'r') as f:
    eval_set = json.load(f)

correct = 0
for item in eval_set:
    result = answer(item["question"], docs)
    if item["answerable"]:
        if item["expected_doc_ids"][0] in result:
            correct += 1
            
    else:
        if "don't have a reliable source" in result:
            correct += 1
            
print(f"{correct}/{len(eval_set)} correct")

8/10 correct


In [16]:
print(answer("What is Aarav Steel's Q3FY28 revenue guidance?", docs))

I don't have a reliable source for this.


In [17]:
# let us build LLM for better answers

def build_prompt(question, results):
    context = "\n".join(f"[{doc['doc_id']}] {doc['text']}" for doc, score in results)
    return (
        "Answer the question using ONLY the sources below. "
        "End every sentence with the matching [doc_id] tag. "
        "If the sources don't answer the question, say exactly: "
        "\"I don't have a reliable source for this.\"\n\n"
        f"Sources:\n{context}\n\nQuestion: {question}\nAnswer:"
    )

In [18]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

In [33]:
from huggingface_hub import InferenceClient
client = InferenceClient(
    token= HF_TOKEN
)

In [36]:

def answer_llm(query, docs, top_k=3):
    results = search(query, docs, top_k)
    if not results:
        return "I don't have a reliable source for this."

    q_quarter = extract_quarter(query)
    top_doc, top_score = results[0]
    if q_quarter and extract_quarter(top_doc.get("quarter", "")) != q_quarter:
        top_score *= 0.4
    if top_score < THERSHOLD:
        return "I don't have a reliable source for this."

    prompt = build_prompt(query, results)
    reply = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V4.1-Flash:novita",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=250,
        temperature=0,
    )
    return reply.choices[0].message.content

In [37]:
print(answer_llm("What was Aarav Steel's revenue?", docs))

Aarav Steel Ltd reported consolidated revenue of INR 4,812 crore for Q1FY27. [ARS-Q2FY26-RES-01]
